# ARC-Jeni-2026 — Hybrid DSA + Program Synthesis + Chakra-Ensemble LLM

**Competition:** ARC Prize 2026 – ARC-AGI-2  
**Author:** joyjeni  
**Repo:** https://github.com/joyjeni/arc-agi2-hybrid-solver  

## Architecture (3-tier)
| Tier | Method | Time Budget | Expected Coverage |
|------|--------|-------------|-------------------|
| 1 | DSA + Vedic deterministic solvers | 300 s total / 5 s task | 5–10% |
| 2 | LLM Program Synthesis (write+run Python) | 40% remaining / 45 s task | 15–25% |
| 3 | Chakra-8 Ensemble direct prediction | remainder / 90 s task | 10–20% |

**Model:** Qwen3-4B fine-tuned on ARC grids (`qwen3_4b_grids15_sft139`)  
**Hard time control:** Tier-1 ThreadPoolExecutor wall-clock; Tier-2/3 `signal.SIGALRM` per task  
**Total wall-clock target:** ≤5 h (18 000 s)

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
import subprocess, sys

_pkgs = [
    'torch', 'transformers', 'accelerate', 'peft', 'pyyaml', 'numpy',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + _pkgs,
    check=True,
)
print('Dependencies ready.')

In [ ]:
# ── Setup & Imports ─────────────────────────────────────────────────────────
import os, sys, time, json, logging, signal, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait as _fut_wait

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('arc_jeni')

IS_KAGGLE = Path('/kaggle').exists()
IS_RERUN  = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

DATA_DIR   = Path('/kaggle/input/arc-prize-2026-arc-agi-2') if IS_KAGGLE else Path('data')
MODEL_DIR  = Path('/kaggle/input/qwen3-4b-grids15-sft139/transformers/sft139/1') if IS_KAGGLE else None
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.monotonic()   # competition start timestamp

print(f'IS_KAGGLE={IS_KAGGLE}  IS_RERUN={IS_RERUN}')
print(f'DATA_DIR={DATA_DIR}  MODEL_DIR={MODEL_DIR}')

In [ ]:
# ── Time budget constants ────────────────────────────────────────────────────
# All values in seconds. Tune here; rest of notebook reads these names.

COMPETITION_LIMIT_S  = 5 * 3600    # 5 h hard wall (Kaggle GPU session)
MARGIN_S             = 300          # 5 min I/O + submission write buffer

# Tier-1: fast DSA solvers run in parallel
T1_BUDGET_S          = 300          # hard cap for ALL Tier-1 (wall-clock)
T1_PER_TASK_S        = 5.0          # max slot per task inside ThreadPoolExecutor

# Tier-2: program synthesis via LLM
T2_SHARE             = 0.40         # fraction of remaining budget after T1
T2_MAX_BUDGET_S      = 7200.0       # 2 h absolute cap for entire Tier-2
T2_MAX_PER_TASK_S    = 45.0         # max slot per task (SIGALRM enforced)
T2_MIN_PER_TASK_S    = 10.0         # quit Tier-2 if fair-share drops below this

# Tier-3: Chakra ensemble direct prediction
T3_MAX_PER_TASK_S    = 90.0         # max slot per task (SIGALRM enforced)
T3_MIN_PER_TASK_S    = 20.0         # quit Tier-3 if fair-share drops below this

def _remaining() -> float:
    """Seconds left before hitting the hard competition wall.""",
    return max(0.0, COMPETITION_LIMIT_S - (time.monotonic() - T0) - MARGIN_S)

print(f'Competition budget: {COMPETITION_LIMIT_S/3600:.1f}h  margin: {MARGIN_S}s')
print(f'T1 budget: {T1_BUDGET_S}s ({T1_PER_TASK_S}s/task)  ',
      f'T2 max: {T2_MAX_BUDGET_S}s ({T2_MAX_PER_TASK_S}s/task)  ',
      f'T3 max/task: {T3_MAX_PER_TASK_S}s')

In [ ]:
# ── Clone repo & add to path ────────────────────────────────────────────────
REPO_URL = 'https://github.com/joyjeni/arc-agi2-hybrid-solver.git'
REPO_DIR = Path('/kaggle/working/arc-agi2-hybrid-solver') if IS_KAGGLE else Path('/home/sandbox/arc-jeni-2026')

if IS_KAGGLE and not (REPO_DIR / 'src').exists():
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)],
        check=False,
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready at', REPO_DIR)
print('src exists:', (REPO_DIR / 'src').exists())

In [ ]:
# ── Load challenge data ─────────────────────────────────────────────────────
from src.pipeline import load_challenges

# Primary: test set (competition scoring)
CHALLENGES_FILE = DATA_DIR / 'arc-agi_test_challenges.json'
if not CHALLENGES_FILE.exists():
    # Fallback: evaluation set for local dev / notebook testing
    CHALLENGES_FILE = DATA_DIR / 'arc-agi_evaluation_challenges.json'
    logger.warning('Test challenges not found — using evaluation set as dev fallback')

tasks = load_challenges(CHALLENGES_FILE)
print(f'Loaded {len(tasks)} tasks from {CHALLENGES_FILE.name}')
print('Sample task IDs:', list(tasks.keys())[:5])
print(f'Remaining budget at data load: {_remaining()/3600:.2f}h')

In [ ]:
# ── Tier-1: DSA + Vedic deterministic solvers (parallel) ────────────────────
from src.solvers import REGISTRY
from src.pipeline import run_tier1

print(f'Registered solvers ({len(REGISTRY.names())}): {REGISTRY.names()[:10]} ...')

t1_start = time.monotonic()
t1_results = run_tier1(
    tasks, set(), REGISTRY,
    total_budget_s=T1_BUDGET_S,
    per_task_s=T1_PER_TASK_S,
)
t1_elapsed = time.monotonic() - t1_start
solved_ids = set(t1_results.keys())
pct = 100 * len(solved_ids) / max(len(tasks), 1)
print(f'Tier-1 done in {t1_elapsed:.1f}s — solved {len(solved_ids)}/{len(tasks)} ({pct:.1f}%)')
print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── LLM Setup: Qwen3-4B from Kaggle model hub ───────────────────────────────
from src.llm.engine import build_engine

model_path = str(MODEL_DIR) if (MODEL_DIR and MODEL_DIR.exists()) else None
if model_path is None:
    logger.warning('Model dir not found — using mock backend (T2/T3 will be no-ops in dev)')

llm_engine = build_engine(model_path=model_path)
print('LLM engine ready:', type(llm_engine).__name__)

In [ ]:
# ── Per-task hard-interrupt handler (SIGALRM — Linux/Kaggle only) ───────────
# signal.SIGALRM only works on the main thread; notebooks run on main thread.

def _on_task_timeout(signum, frame):
    raise TimeoutError('Hard per-task timeout (SIGALRM)')

# Verify signal module is available
assert hasattr(signal, 'SIGALRM'), 'SIGALRM not available on this platform'
print('SIGALRM available — hard per-task timeout enabled')

In [ ]:
# ── Tier-2: Program Synthesis with hard per-task SIGALRM timeout ────────────
from src.llm.program_synth import ProgramSynthesisSolver

t2_remaining_budget = min(T2_SHARE * _remaining(), T2_MAX_BUDGET_S)
t2_pending = [(tid, ctx) for tid, ctx in tasks.items() if tid not in solved_ids]
n_t2 = len(t2_pending)
print(f'Tier-2: {n_t2} tasks | budget={t2_remaining_budget:.0f}s | max/task={T2_MAX_PER_TASK_S}s')

t2_results: dict = {}
t2_solver = ProgramSynthesisSolver(llm_engine)
_old_t2_handler = signal.signal(signal.SIGALRM, _on_task_timeout)
t2_loop_start = time.monotonic()

for _i, (_tid, _ctx) in enumerate(t2_pending):
    _t2_elapsed = time.monotonic() - t2_loop_start
    _t2_left    = t2_remaining_budget - _t2_elapsed
    _tasks_left = n_t2 - _i
    if _t2_left < 30.0:
        logger.info(f'T2: budget exhausted at task {_i}/{n_t2} — stopping')
        break
    _fair_share = _t2_left / _tasks_left
    _per_task   = min(T2_MAX_PER_TASK_S, _fair_share)
    if _per_task < T2_MIN_PER_TASK_S:
        logger.info(f'T2: fair-share {_per_task:.1f}s < min {T2_MIN_PER_TASK_S}s — stopping')
        break
    # Set hard per-task alarm
    signal.alarm(max(1, int(_per_task)))
    try:
        _res = t2_solver._timed_solve(_ctx)
        signal.alarm(0)   # cancel alarm on success
        if _res and _res.solved:
            t2_results[_tid] = _res
    except TimeoutError:
        signal.alarm(0)
        logger.warning(f'T2: {_tid} hard-timeout after {_per_task:.1f}s')
    except Exception as _exc:
        signal.alarm(0)
        logger.debug(f'T2: {_tid} error: {_exc}')
    if (_i + 1) % 10 == 0:
        logger.info(f'T2 progress {_i+1}/{n_t2}: {len(t2_results)} solved | {_t2_left:.0f}s left')

signal.signal(signal.SIGALRM, _old_t2_handler)  # restore
solved_ids |= set(t2_results.keys())
t2_elapsed_total = time.monotonic() - t2_loop_start
print(f'Tier-2 done in {t2_elapsed_total:.1f}s — solved {len(t2_results)}/{n_t2} new tasks')
print(f'Cumulative solved: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/max(len(tasks),1):.1f}%)')
print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── Tier-3: Chakra Ensemble with hard per-task SIGALRM timeout ──────────────
from src.llm.chakra_ensemble import ChakraEnsembleSolver

t3_budget = max(0.0, _remaining())
t3_pending = [(tid, ctx) for tid, ctx in tasks.items() if tid not in solved_ids]
n_t3 = len(t3_pending)
print(f'Tier-3: {n_t3} tasks | budget={t3_budget:.0f}s ({t3_budget/3600:.2f}h) | max/task={T3_MAX_PER_TASK_S}s')

t3_results: dict = {}
_old_t3_handler = signal.signal(signal.SIGALRM, _on_task_timeout)
t3_loop_start = time.monotonic()

for _i, (_tid, _ctx) in enumerate(t3_pending):
    _t3_elapsed = time.monotonic() - t3_loop_start
    _t3_left    = t3_budget - _t3_elapsed
    _tasks_left = n_t3 - _i
    if _t3_left < 30.0:
        logger.info(f'T3: budget exhausted at task {_i}/{n_t3} — stopping')
        break
    _fair_share = _t3_left / _tasks_left
    _per_task   = min(T3_MAX_PER_TASK_S, _fair_share)
    if _per_task < T3_MIN_PER_TASK_S:
        logger.info(f'T3: fair-share {_per_task:.1f}s < min {T3_MIN_PER_TASK_S}s — stopping')
        break
    # Rebuild solver per task so per_task_timeout is always tight
    _solver = ChakraEnsembleSolver(llm_engine, n_templates=4, per_task_timeout=float(_per_task))
    signal.alarm(max(1, int(_per_task) + 5))  # +5s grace for thread cleanup
    try:
        _res = _solver._timed_solve(_ctx)
        signal.alarm(0)
        if _res and _res.solved:
            t3_results[_tid] = _res
    except TimeoutError:
        signal.alarm(0)
        logger.warning(f'T3: {_tid} hard-timeout after {_per_task:.1f}s')
    except Exception as _exc:
        signal.alarm(0)
        logger.debug(f'T3: {_tid} error: {_exc}')
    if (_i + 1) % 20 == 0:
        logger.info(f'T3 progress {_i+1}/{n_t3}: {len(t3_results)} solved | {_t3_left:.0f}s left')

signal.signal(signal.SIGALRM, _old_t3_handler)  # restore
solved_ids |= set(t3_results.keys())
t3_elapsed_total = time.monotonic() - t3_loop_start
print(f'Tier-3 done in {t3_elapsed_total:.1f}s — solved {len(t3_results)}/{n_t3} new tasks')
print(f'Final solved: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/max(len(tasks),1):.1f}%)')
print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── Build & validate submission ─────────────────────────────────────────────
from src.core.submission import build_submission, validate_submission, write_submission

all_results = {**t1_results, **t2_results, **t3_results}
submission = build_submission(tasks, all_results, fallback_input=True)

errors = validate_submission(submission, tasks)
if errors:
    for e in errors[:10]:
        print('VALIDATION ERROR:', e)
    if len(errors) > 10:
        print(f'  ... and {len(errors)-10} more')
else:
    print(f'Submission valid: {len(submission)} tasks')

# Sample entry check
sample_tid = list(submission.keys())[0]
sample_pred = submission[sample_tid][0]
a1, a2 = sample_pred['attempt_1'], sample_pred['attempt_2']
print(f'Sample [{sample_tid}]: attempt_1 {len(a1)}x{len(a1[0])}  attempt_2 {len(a2)}x{len(a2[0])}')

In [ ]:
# ── Write submission.json ────────────────────────────────────────────────────
OUT_PATH = OUTPUT_DIR / 'submission.json'
write_submission(submission, OUT_PATH, validate=False)

# Verify written file
_loaded = json.loads(OUT_PATH.read_text())
assert len(_loaded) == len(tasks), f'Task count mismatch: {len(_loaded)} vs {len(tasks)}'
for _tid, _preds in _loaded.items():
    for _pred in _preds:
        assert 'attempt_1' in _pred and 'attempt_2' in _pred, f'Missing attempt keys in {_tid}'

total_elapsed = time.monotonic() - T0
_tier_counts = {
    1: sum(1 for r in all_results.values() if r.tier == 1),
    2: sum(1 for r in all_results.values() if r.tier == 2),
    3: sum(1 for r in all_results.values() if r.tier == 3),
}
_fallback = len(tasks) - len(all_results)

print(f'submission.json written: {OUT_PATH}')
print(f'Total tasks : {len(tasks)}')
print(f'Tier-1 (DSA)       : {_tier_counts[1]}')
print(f'Tier-2 (ProgSynth) : {_tier_counts[2]}')
print(f'Tier-3 (Chakra LLM): {_tier_counts[3]}')
print(f'Fallback (input copy): {_fallback}')
print(f'Elapsed : {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f}h)')
print(f'Budget used: {100*total_elapsed/COMPETITION_LIMIT_S:.1f}%')